# 04 · Kaggle held-out evaluation

Attach the saved output from notebook 03. Evaluate the official held-out test set only once after model selection.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/mamta-kumari28/Chest-X-Ray-Diagnosis-using-Deep-Learning-.git"
REPO_REF = "phase2/final-validation-fixes"  # Change to main after the PR is merged.
ROOT = Path("/kaggle/working/covid-xray-project")
OUTPUT_ROOT = Path("/kaggle/working/covid_xray_outputs")

if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, "--depth", "1", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Outputs:", OUTPUT_ROOT)

In [ ]:
def find_one(name, roots=(Path("/kaggle/working"), Path("/kaggle/input"))):
    matches = []
    for root in roots:
        matches.extend(root.glob(f"**/{name}"))
    matches = [path for path in matches if ".git" not in path.parts]
    if not matches:
        raise FileNotFoundError(f"Could not find {name}. Attach the previous notebook output as a Kaggle input.")
    print(name, "->", matches[0])
    return matches[0]

MANIFEST = find_one("manifest.csv")
MODEL = find_one("model.keras")

In [ ]:
RESULTS = OUTPUT_ROOT / "evaluation"
subprocess.run([
    sys.executable, "-m", "covid_xray.evaluate",
    "--manifest", str(MANIFEST), "--model", str(MODEL),
    "--output", str(RESULTS), "--batch-size", "32",
], check=True)

In [ ]:
import json
from IPython.display import Image as DisplayImage, display

metrics = json.loads((RESULTS / "metrics.json").read_text())
display(metrics)
for name in ("confusion_matrix.png", "roc_curves.png", "error_analysis.png"):
    path = RESULTS / name
    if path.exists():
        display(DisplayImage(filename=str(path)))
print("Evaluation outputs:", RESULTS)
print("Experimental research output; not a medical diagnosis.")

Use **Save Version → Save & Run All** and download the executed notebook plus `covid_xray_outputs`. Upload them for repository review; large/private artifacts will remain outside Git.